In [2]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .appName("Week7_Delta_Lake") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("Spark Version:", spark.version)

:: loading settings :: url = jar:file:/Users/prasidhyikumar/Library/Python/3.9/lib/python/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/prasidhyikumar/.ivy2/cache
The jars for the packages stored in: /Users/prasidhyikumar/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1ba8024f-84db-4455-b911-08260b7e2cdc;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 67ms :: artifacts dl 2ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default  

Spark Version: 3.5.9


# Week 7 - Delta Lake Incremental Data Processing

## Objective

The objective of this assignment is to perform incremental data processing using Delta Lake by loading customer data, cleaning the dataset, applying MERGE operations to update existing records and insert new records, validating the results, and displaying the final Delta table.

In [3]:
master_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("../data/customer_master.csv")

master_df.show()

+-----------+-------------+---------+-----------------+--------+
|customer_id|customer_name|     city|            email|  status|
+-----------+-------------+---------+-----------------+--------+
|        101|        Alice|    Delhi|  alice@gmail.com|  Active|
|        102|          Bob|   Mumbai|    bob@gmail.com|  Active|
|        103|      Charlie|     Pune|charlie@gmail.com|Inactive|
|        104|        David|   Jaipur|  david@gmail.com|  Active|
|        105|         Emma|    Delhi|   emma@gmail.com|  Active|
|        106|        Frank|Hyderabad|             NULL|  Active|
|        107|        Grace|  Chennai|  grace@gmail.com|  Active|
|        107|        Grace|  Chennai|  grace@gmail.com|  Active|
+-----------+-------------+---------+-----------------+--------+



In [4]:
print("Number of Rows:", master_df.count())
print("Number of Columns:", len(master_df.columns))

master_df.printSchema()

master_df.show(5)

Number of Rows: 8
Number of Columns: 5
root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- email: string (nullable = true)
 |-- status: string (nullable = true)

+-----------+-------------+------+-----------------+--------+
|customer_id|customer_name|  city|            email|  status|
+-----------+-------------+------+-----------------+--------+
|        101|        Alice| Delhi|  alice@gmail.com|  Active|
|        102|          Bob|Mumbai|    bob@gmail.com|  Active|
|        103|      Charlie|  Pune|charlie@gmail.com|Inactive|
|        104|        David|Jaipur|  david@gmail.com|  Active|
|        105|         Emma| Delhi|   emma@gmail.com|  Active|
+-----------+-------------+------+-----------------+--------+
only showing top 5 rows



In [5]:
clean_df = (
    master_df
    .dropDuplicates()
    .na.drop(subset=["email"])
)

clean_df.show()

+-----------+-------------+-------+-----------------+--------+
|customer_id|customer_name|   city|            email|  status|
+-----------+-------------+-------+-----------------+--------+
|        107|        Grace|Chennai|  grace@gmail.com|  Active|
|        103|      Charlie|   Pune|charlie@gmail.com|Inactive|
|        102|          Bob| Mumbai|    bob@gmail.com|  Active|
|        104|        David| Jaipur|  david@gmail.com|  Active|
|        105|         Emma|  Delhi|   emma@gmail.com|  Active|
|        101|        Alice|  Delhi|  alice@gmail.com|  Active|
+-----------+-------------+-------+-----------------+--------+



In [10]:
clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("../data/customer_delta")

In [11]:
delta_df = spark.read \
    .format("delta") \
    .load("../data/customer_delta")

delta_df.show()

+-----------+-------------+-------+-----------------+--------+
|customer_id|customer_name|   city|            email|  status|
+-----------+-------------+-------+-----------------+--------+
|        107|        Grace|Chennai|  grace@gmail.com|  Active|
|        103|      Charlie|   Pune|charlie@gmail.com|Inactive|
|        102|          Bob| Mumbai|    bob@gmail.com|  Active|
|        104|        David| Jaipur|  david@gmail.com|  Active|
|        105|         Emma|  Delhi|   emma@gmail.com|  Active|
|        101|        Alice|  Delhi|  alice@gmail.com|  Active|
+-----------+-------------+-------+-----------------+--------+



In [13]:
incremental_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("../data/customer_incremental.csv")

incremental_df.show()

+-----------+-------------+---------+------------------+--------+
|customer_id|customer_name|     city|             email|  status|
+-----------+-------------+---------+------------------+--------+
|        103|      Charlie|Bangalore| charlie@gmail.com|  Active|
|        105|         Emma|    Delhi|emma_new@gmail.com|  Active|
|        108|        Harry|  Kolkata|   harry@gmail.com|  Active|
|        109|         Isha|Ahmedabad|    isha@gmail.com|Inactive|
+-----------+-------------+---------+------------------+--------+



In [14]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, "../data/customer_delta")

(
    delta_table.alias("target")
    .merge(
        incremental_df.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdate(
        set={
            "customer_name": "source.customer_name",
            "city": "source.city",
            "email": "source.email",
            "status": "source.status"
        }
    )
    .whenNotMatchedInsert(
        values={
            "customer_id": "source.customer_id",
            "customer_name": "source.customer_name",
            "city": "source.city",
            "email": "source.email",
            "status": "source.status"
        }
    )
    .execute()
)

In [15]:
final_df = spark.read \
    .format("delta") \
    .load("../data/customer_delta")

final_df.show()

+-----------+-------------+---------+------------------+--------+
|customer_id|customer_name|     city|             email|  status|
+-----------+-------------+---------+------------------+--------+
|        101|        Alice|    Delhi|   alice@gmail.com|  Active|
|        102|          Bob|   Mumbai|     bob@gmail.com|  Active|
|        103|      Charlie|Bangalore| charlie@gmail.com|  Active|
|        104|        David|   Jaipur|   david@gmail.com|  Active|
|        105|         Emma|    Delhi|emma_new@gmail.com|  Active|
|        107|        Grace|  Chennai|   grace@gmail.com|  Active|
|        108|        Harry|  Kolkata|   harry@gmail.com|  Active|
|        109|         Isha|Ahmedabad|    isha@gmail.com|Inactive|
+-----------+-------------+---------+------------------+--------+



In [17]:
print("Final Row Count:", final_df.count())

print("Duplicate Customer IDs:")

final_df.groupBy("customer_id") \
    .count() \
    .filter("count > 1") \
    .show()

Final Row Count: 8
Duplicate Customer IDs:
+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



# Summary

- Loaded the customer master dataset into a Delta table.
- Cleaned the data by removing duplicate records and rows with missing email values.
- Loaded incremental customer data.
- Used Delta Lake MERGE to update existing customer records and insert new customer records.
- Validated the final dataset by checking the total row count and ensuring there were no duplicate customer IDs.
- Successfully demonstrated incremental data processing using Delta Lake.